In [ ]:
# Libs
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime, timedelta
from collections import defaultdict
from skimage.measure import label, regionprops
import croco_plot as cplot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
%matplotlib inline


In [ ]:
# Params

# Create an empty list to store structure data
structure_data = []

# Parameters
min_vort = 1.5e-5
min_area = 20

# Load data
data_path = '/lus/work/CT1/c1601279/rguillermin/RUN_CROCO/run_swio2_deter2_2017_2023/swio_avg_suf.nc'
grid_path = '/lus/work/CT1/c1601279/lweiss/CROCO/RUN/SWIOSE/CROCO_FILES/grid/croco_grid_swio2.nc'

lon, lat, pm, pn, msk, msk_inv, angle, _ = cplot.utils.load_grid(grid_path)
u_geo, v_geo, vorticity = cplot.utils.load_data(data_path, ('u_geo', 'v_geo', 'vort'))

u_geo, v_geo, vorticity = u_geo[:, 0, :, :], v_geo[:, 0, :, :], vorticity[:, 0, :, :]

gridline_style = {'draw_labels': True, 'linestyle': '--', 'linewidth': 0.3}

day_seconds = 24 * 60 * 60


In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime
import pandas as pd  # Optional for datetime handling

class Eddy:
    def __init__(self, date, area, perimeter, centroid_lon, centroid_lat, mean_value, mean_u, mean_v):
        if isinstance(date, str):
            self.date = datetime.strptime(date, '%Y-%m-%d')  # Parse string to datetime
        elif isinstance(date, datetime):
            self.date = date
        else:
            raise ValueError("date must be a datetime object or string in 'YYYY-MM-DD' format.")
        
        self.area = area
        self.perimeter = perimeter
        self.centroid = (centroid_lon, centroid_lat)
        self.vorticity = mean_value
        self.velocity = (mean_u, mean_v)

    def __repr__(self):
        return (f"Eddy: (date={self.date.strftime('%Y-%m-%d')}, area={int(self.area)}, perimeter={int(self.perimeter)}, "
                f"centroid={np.round(self.centroid, 2)}, vorticity={np.round(self.vorticity, 6)}, "
                f"velocity={np.round(self.velocity, 3)})")

def save_eddies_to_netcdf(eddies, filename):
    """
    Save a list of Eddy objects to a NetCDF file via xarray.Dataset.
    """
    # Collect attributes into lists
    data_dict = {
        "date": [],
        "area": [],
        "perimeter": [],
        "centroid_lon": [],
        "centroid_lat": [],
        "mean_value": [],
        "mean_u": [],
        "mean_v": []
    }

    for eddy in eddies:
        data_dict["date"].append(eddy.date.strftime('%Y-%m-%d'))  # Save as string
        data_dict["area"].append(eddy.area)
        data_dict["perimeter"].append(eddy.perimeter)
        data_dict["centroid_lon"].append(eddy.centroid[0])
        data_dict["centroid_lat"].append(eddy.centroid[1])
        data_dict["mean_value"].append(eddy.vorticity)
        data_dict["mean_u"].append(eddy.velocity[0])
        data_dict["mean_v"].append(eddy.velocity[1])
    
    # Create xarray Dataset
    ds = xr.Dataset(
        {
            "area": ("region", data_dict["area"]),
            "perimeter": ("region", data_dict["perimeter"]),
            "centroid_lon": ("region", data_dict["centroid_lon"]),
            "centroid_lat": ("region", data_dict["centroid_lat"]),
            "mean_value": ("region", data_dict["mean_value"]),
            "mean_u": ("region", data_dict["mean_u"]),
            "mean_v": ("region", data_dict["mean_v"]),
        },
        coords={
            "region": np.arange(len(eddies)),
            "date": ("region", data_dict["date"])  # Saved as string
        }
    )
    
    # Save to NetCDF
    ds.to_netcdf(filename)
    print(f"Saved {len(eddies)} regions to '{filename}'")

def load_eddies_from_netcdf(filename):
    """
    Load Eddy objects from a NetCDF file created by `save_eddies_to_netcdf`.
    """
    ds = xr.open_dataset(filename)
    eddies = []
    
    for i in range(len(ds.region)):
        date_str = str(ds.date[i].item())  # Extract as string
        date = datetime.strptime(date_str, '%Y-%m-%d')  # Convert to datetime
        area = ds.area[i].item()
        perimeter = ds.perimeter[i].item()
        centroid_lon = ds.centroid_lon[i].item()
        centroid_lat = ds.centroid_lat[i].item()
        mean_value = ds.mean_value[i].item()
        mean_u = ds.mean_u[i].item()
        mean_v = ds.mean_v[i].item()
        
        region = Eddy(
            date=date,
            area=area,
            perimeter=perimeter,
            centroid_lon=centroid_lon,
            centroid_lat=centroid_lat,
            mean_value=mean_value,
            mean_u=mean_u,
            mean_v=mean_v
        )
        eddies.append(region)
    
    print(f"Loaded {len(eddies)} regions from '{filename}'")
    return eddies


In [ ]:
eddies = []

# Loop over time indices
for index in range(5):
    date = np.datetime_as_string(vorticity.time.data[index], 'D')
    
    data = vorticity[index].data
    u_data, v_data = u_geo[index].data, v_geo[index].data 
    data[np.abs(data) > 1e10] = 0  # Remove extreme values
    
    # Masking
    mask_values = np.zeros_like(data, dtype=int)
    mask_values[data > min_vort] = 1   # Positive vorticity
    mask_values[data < -min_vort] = -1 # Negative vorticity

    # Process positive and negative vorticity separately
    for sign, label_value in zip([1, -1], ["positive", "negative"]):
        region_mask = (mask_values == sign)
        labeled_regions = label(region_mask)
        
        # Extract region properties using skimage
        regions = regionprops(labeled_regions, intensity_image=data)

        for region in regions:
            if region.area >= min_area:
                # Convert centroid indices to lat/lon
                centroid_x, centroid_y = region.centroid  # Skimage returns (row, col)
                centroid_lon = lon.data[int(centroid_x), int(centroid_y)]
                centroid_lat = lat.data[int(centroid_x), int(centroid_y)]
                
                # Compute mean velocity inside the detected structure
                mean_u = np.mean(u_data[labeled_regions == region.label])
                mean_v = np.mean(v_data[labeled_regions == region.label])
                
                eddy = Eddy(date, region.area, region.perimeter, centroid_lon, centroid_lat, region.mean_intensity, mean_u, mean_v )
                
                eddies.append(eddy)


save_eddies_to_netcdf(eddies, "eddies.nc")


In [ ]:
loaded_eddies = load_eddies_from_netcdf("eddies.nc")

date_index = defaultdict(list)
for eddy in loaded_eddies:
    date_index[eddy.date].append(eddy)
    
trajectories = []  # List of trajectories (each trajectory is a list of eddies)

# Step 1: Initialize trajectories with eddies from the first date
all_dates = sorted(set(eddy.date for eddy in loaded_eddies))
first_date = all_dates[0]
eddies_first_day = date_index[first_date]

for eddy in eddies_first_day:
    trajectories.append([eddy])  # Start each trajectory

# Step 2: Iteratively link eddies from day to day
for i in range(len(all_dates) - 1):
    current_date = all_dates[i]
    next_date = all_dates[i + 1]

    current_eddies = date_index[current_date]
    next_eddies = date_index[next_date]

    # Keep track of which eddies are already matched
    matched_next_eddies = set()

    for traj in trajectories:
        last_eddy = traj[-1]  # Last eddy in the current trajectory

        # Calculate allowed distance
        dist = np.sqrt(last_eddy.velocity[0]**2 + last_eddy.velocity[1]**2) * day_seconds / 1000  # km
        lat_dist = (dist / 111.32) * 2  # degrees latitude (scaled x2)
        lon_dist = (dist / (111.32 * np.cos(np.deg2rad(last_eddy.centroid[1])))) * 2  # degrees longitude (scaled x2)
        deg_dist = lat_dist**2 + lon_dist**2
        
        vorticity_sign = np.sign(last_eddy.vorticity)

        # Find candidate matches
        candidates = []
        for idx, next_eddy in enumerate(next_eddies):
            if idx in matched_next_eddies:
                continue  # Skip if already matched to another trajectory

            lat_diff = np.abs(next_eddy.centroid[1] - last_eddy.centroid[1])
            lon_diff = np.abs(next_eddy.centroid[0] - last_eddy.centroid[0])
            next_sign = np.sign(next_eddy.vorticity)

            if (lat_diff ** 2 + lon_diff ** 2 < deg_dist) and (vorticity_sign == next_sign):
                candidates.append((idx, next_eddy))

        # If candidate found, pick the closest (or first) and add to trajectory
        if candidates:
            idx_match, eddy_match = candidates[0]  # You can improve selection criteria here (e.g., minimum distance)
            traj.append(eddy_match)
            matched_next_eddies.add(idx_match)

# Step 3: Filter out short trajectories (optional)
min_length = 1  # Minimum number of eddies to be considered a trajectory
filtered_trajectories = [traj for traj in trajectories if len(traj) >= min_length]

print(f"Found {len(filtered_trajectories)} trajectories with at least {min_length} steps.")



In [ ]:
# Assuming `trajectories` is a list of lists (each list contains eddies forming a trajectory)

# Prepare colors: generate a colormap for number of trajectories
num_trajectories = len(trajectories)
cmap = plt.get_cmap('viridis', num_trajectories)
norm = mcolors.Normalize(vmin=0, vmax=num_trajectories - 1)

# Create the figure
fig, ax = plt.subplots(figsize=(20, 12), subplot_kw={'projection': ccrs.PlateCarree()})

# Set extent based on all eddy positions
all_lons = [eddy.centroid[0] for traj in trajectories for eddy in traj]
all_lats = [eddy.centroid[1] for traj in trajectories for eddy in traj]
ax.set_extent([min(all_lons)-1, max(all_lons)+1, min(all_lats)-1, max(all_lats)+1])

# Plot trajectories
for idx, traj in enumerate(trajectories):
    traj_lons = [eddy.centroid[0] for eddy in traj]
    traj_lats = [eddy.centroid[1] for eddy in traj]
    traj_dates = [eddy.date for eddy in traj]  # If you want to label or color-code

    # Plot line trajectory
    ax.plot(traj_lons, traj_lats, '-', color=cmap(norm(idx)), linewidth=2, alpha=0.8)

    # Optional: Plot start and end points
    ax.scatter(traj_lons[0], traj_lats[0], color=cmap(norm(idx)), edgecolor='black', s=30, label=f'Traj {idx+1} start')
    ax.scatter(traj_lons[-1], traj_lats[-1], color=cmap(norm(idx)), edgecolor='white', s=30, label=f'Traj {idx+1} end')

# Plot background contour (land mask or similar)
ax.contourf(lon, lat, msk_inv, colors='lightgray')
ax.contour(lon, lat, msk, colors='k', linewidths=0.2)

# Add gridlines and styling
gl = ax.gridlines(crs=ccrs.PlateCarree(), **gridline_style)
gl.bottom_labels = False
gl.left_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 8, 'color': 'k'}

# ---- Add inset axes ----
axins = ax.inset_axes([0.5,0.03, 0.47, 0.47], projection = ccrs.PlateCarree(), anchor='SE')
axins.set_extent(ax.get_extent(), crs=ccrs.PlateCarree())

# Plot trajectories
for idx, traj in enumerate(trajectories):
    traj_lons = [eddy.centroid[0] for eddy in traj]
    traj_lats = [eddy.centroid[1] for eddy in traj]
    traj_dates = [eddy.date for eddy in traj]  # If you want to label or color-code

    # Plot line trajectory
    axins.plot(traj_lons, traj_lats, '-', color=cmap(norm(idx)), linewidth=2, alpha=0.8)
    
    # Optional: Plot start and end points
    axins.scatter(traj_lons[0], traj_lats[0], color=cmap(norm(idx)), edgecolor='black', s=30, label=f'Traj {idx+1} start')
    axins.scatter(traj_lons[-1], traj_lats[-1], color=cmap(norm(idx)), edgecolor='white', s=30, label=f'Traj {idx+1} end')

# ---- Plot background contour (land/sea mask) ----
axins.contourf(lon, lat, msk_inv, colors='lightgray')
axins.contour(lon, lat, msk, colors='k', linewidths=0.5)

axins.set_xlim((40, 47))
axins.set_ylim((-17, -10))

ind_zoom = ax.indicate_inset_zoom(axins, edgecolor='black', alpha=1)

# Title and labels
ax.set_title("Eddy Trajectories Over Time")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Show plot
plt.show()


In [ ]:
# ---- Create figure and axis ----
fig, axes = plt.subplots(1, 2, figsize=(20, 8), subplot_kw={'projection': ccrs.PlateCarree()})

ax = axes[0]

# ---- Extract longitude, latitude, and date ----
all_lons = [eddy.centroid[0] for eddy in loaded_eddies]
all_lats = [eddy.centroid[1] for eddy in loaded_eddies]
all_dates = [eddy.date for eddy in loaded_eddies]

all_dist = np.array([np.sqrt(eddy.velocity[0]**2 + eddy.velocity[1]**2) * day_seconds / 1000 for eddy in loaded_eddies])
all_lat_dist = (dist / 111.32) 
all_lon_dist = np.array([(dist / (111.32 * np.cos(np.deg2rad(eddy.centroid[1]))))for eddy in loaded_eddies])
all_deg_dist = np.sqrt(all_lat_dist ** 2 + all_lon_dist ** 2) * 2

all_sign = [np.sign(eddy.vorticity) for eddy in loaded_eddies]

# ---- Convert dates to matplotlib numeric format for coloring ----
date_nums = mdates.date2num(all_dates)

# ---- Set extent slightly padded ----
pad = 1  # degrees padding
ax.set_extent([
    min(all_lons) - pad, max(all_lons) + pad,
    min(all_lats) - pad, max(all_lats) + pad
])

# ---- Scatter plot: Centroids with color mapped to date ----
sc = ax.scatter(all_lons, all_lats, c=date_nums, cmap="cividis", s=10, alpha=0.8, edgecolor='k', linewidth=0.2)

# ---- Colorbar with formatted date labels ----
cbar = plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.05)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
cbar.ax.tick_params(labelsize=8)

# ---- Add circles around each eddy (reachable distance) ----
for lon_c, lat_c, radius_deg, sign in zip(all_lons, all_lats, all_deg_dist, all_sign):
    if sign == -1:
        color = 'blue'
    else:
        color = 'red'
    circle = mpatches.Circle(
        (lon_c, lat_c), radius_deg,
        edgecolor=color, facecolor='none', linewidth=0.7, alpha=0.7, linestyle='dashed',
        transform=ccrs.PlateCarree()
    )
    ax.add_patch(circle)

# ---- Plot background contour (land/sea mask) ----
ax.contourf(lon, lat, msk_inv, colors='lightgray')
ax.contour(lon, lat, msk, colors='k', linewidths=0.5)

# ---- Add gridlines ----
gl = ax.gridlines(crs=ccrs.PlateCarree(), **gridline_style)
gl.bottom_labels = False
gl.left_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 8, 'color': 'k'}

# ---- Add inset axes ----
axins = ax.inset_axes([0.5,0.03, 0.47, 0.47], projection = ccrs.PlateCarree(), anchor='SE')
axins.set_extent(ax.get_extent(), crs=ccrs.PlateCarree())

scins = axins.scatter(all_lons, all_lats, c=date_nums, cmap="cividis", s=10, alpha=0.8, edgecolor='k', linewidth=0.2)

for lon_c, lat_c, radius_deg, sign in zip(all_lons, all_lats, all_deg_dist, all_sign):
    if sign == -1:
        color = 'blue'
    else:
        color = 'red'
    circle_ins = mpatches.Circle(
        (lon_c, lat_c), radius_deg,
        edgecolor=color, facecolor='none', linewidth=1, alpha=1, linestyle='dashed',
        transform=ccrs.PlateCarree()
    )
    axins.add_patch(circle_ins)

# ---- Plot background contour (land/sea mask) ----
axins.contourf(lon, lat, msk_inv, colors='lightgray')
axins.contour(lon, lat, msk, colors='k', linewidths=0.5)

axins.set_xlim((40, 45))
axins.set_ylim((-15, -10))

ind_zoom = ax.indicate_inset_zoom(axins, edgecolor='black', alpha=1)

# ---- Title and axis labels ----
ax.set_title("All Detected Eddy Centroids")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax = axes[1]

# ---- Flatten all eddies from trajectories ----
all_eddies = [eddy for traj in trajectories for eddy in traj]

# ---- Extract longitude, latitude, and date ----
all_lons = [eddy.centroid[0] for eddy in all_eddies]
all_lats = [eddy.centroid[1] for eddy in all_eddies]
all_dates = [eddy.date for eddy in all_eddies]

# ---- Convert dates to matplotlib numeric format for coloring ----
date_nums = mdates.date2num(all_dates)

# ---- Set extent slightly padded ----
pad = 1  # degrees padding
ax.set_extent([
    min(all_lons) - pad, max(all_lons) + pad,
    min(all_lats) - pad, max(all_lats) + pad
])

# ---- Scatter plot: Centroids with color mapped to date ----
sc = ax.scatter(all_lons, all_lats, c=date_nums, cmap="cividis", s=10, alpha=0.8, edgecolor='k', linewidth=0.2)

# ---- Colorbar with formatted date labels ----
cbar = plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.05)
cbar.set_label("Date", fontsize=12)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
cbar.ax.tick_params(labelsize=8)

# ---- Plot background contour (land/sea mask) ----
ax.contourf(lon, lat, msk_inv, colors='lightgray')
ax.contour(lon, lat, msk, colors='k', linewidths=0.5)

# ---- Add gridlines ----
gl = ax.gridlines(crs=ccrs.PlateCarree(), **gridline_style)
gl.bottom_labels = False
gl.left_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 8, 'color': 'k'}

# ---- Title and axis labels ----
ax.set_title("All Tracked Eddy Centroids")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# ---- Show plot ----
plt.tight_layout()
plt.show()
